In [ ]:
!pip install gdown
!gdown --id 1LXI6YWPJ3apGyg9l9Boo00AbeDkCzKbk -O train_data.csv
!gdown --id 1wNng9IpTCxIdggr0rp2xOabyyniOWlRJ -O test_data.csv

/usr/local/lib/python3.10/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1LXI6YWPJ3apGyg9l9Boo00AbeDkCzKbk
From (redirected): https://drive.google.com/uc?id=1LXI6YWPJ3apGyg9l9Boo00AbeDkCzKbk&confirm=t&uuid=dfc5d07c-5ea6-4bef-9766-e0267a5f8fa5
To: /content/train_data.csv
100% 412M/412M [00:10<00:00, 40.9MB/s]
/usr/local/lib/python3.10/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1wNng9IpTCxIdggr0rp2xOabyyniOWlRJ
To: /content/test_data.csv
100% 103M/103M [00:02<00:00, 43.0MB/s] 


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, GRU, LSTM, Dense, Dropout

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load dataset
train_data = pd.read_csv('/content/train_data.csv',header=None)
test_data = pd.read_csv('/content/test_data.csv',header=None)

In [ ]:
print(f" Shape of train data: {train_data.shape}")
print(f" shape of test data: {test_data.shape}")

 Shape of train data: (87554, 188)
 shape of test data: (21892, 188)


In [ ]:
train_data.head()

,0,1,2,3,4,5,6,7,8,9,...,178,179,180,181,182,183,184,185,186,187
0,0.977941,0.926471,0.681373,0.245098,0.154412,0.191176,0.151961,0.085784,0.058824,0.049020,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.960114,0.863248,0.461538,0.196581,0.094017,0.125356,0.099715,0.088319,0.074074,0.082621,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.000000,0.659459,0.186486,0.070270,0.070270,0.059459,0.056757,0.043243,0.054054,0.045946,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.925414,0.665746,0.541436,0.276243,0.196133,0.077348,0.071823,0.060773,0.066298,0.058011,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.967136,1.000000,0.830986,0.586854,0.356808,0.248826,0.145540,0.089202,0.117371,0.150235,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
print(f"Missing values in train data: {train_data.isna().sum().sum()}")
print(f"Missing values in test data: {test_data.isna().sum().sum()}")

Missing values in train data: 0
Missing values in test data: 0


In [ ]:
print(f"Duplicated values in train data: {train_data.duplicated().sum()}")
print(f"Duplicated values in test data: {test_data.duplicated().sum()}")

Duplicated values in train data: 0
Duplicated values in test data: 0


In [ ]:
print(train_data.iloc[:, -1].value_counts())  # class distribution in train data

187
0.0    72471
4.0     6431
2.0     5788
1.0     2223
3.0      641
Name: count, dtype: int64


In [ ]:
print(test_data.iloc[:, -1].value_counts())  # class distribution in test data

187
0.0    18118
4.0     1608
2.0     1448
1.0      556
3.0      162
Name: count, dtype: int64


In [ ]:
#     0: "Normal",
#     1: "Artial Premature",
#     2: "Premature ventricular contraction",
#     3: "Fusion of ventricular and normal",
#     4: "Fusion of paced and normal"

In [ ]:
# Separate features and labels
X_train = train_data.iloc[:, :-1].values  # Exclude label
y_train = train_data.iloc[:, -1].values   # Labels

X_test = test_data.iloc[:, :-1].values
y_test = test_data.iloc[:, -1].values

# Scailing the data
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Split the training data into new training and validation sets
X_val, X_test_new, y_val, y_test_new = train_test_split(X_test_scaled,
                                                        y_test, test_size=0.5,
                                                        random_state=42, stratify=y_test)

In [ ]:
# Reshape the data for RNN/LSTM/GRU (samples, timesteps, features)
X_train_scaled = np.reshape(X_train_scaled, (X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_val = np.reshape(X_val, (X_val.shape[0], X_val.shape[1], 1))
X_test_new = np.reshape(X_test_new, (X_test_new.shape[0], X_test_new.shape[1], 1))

# RNN

In [ ]:
def build_rnn_model():
    model = Sequential()
    model.add(SimpleRNN(64, input_shape=(X_train.shape[1], 1), return_sequences=True))
    model.add(Dropout(0.2))
    model.add(SimpleRNN(32))
    model.add(Dropout(0.2))
    model.add(Dense(5, activation='softmax'))

    return model

In [ ]:
rnn_model = build_rnn_model()

In [ ]:
rnn_model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
rnn_history = rnn_model.fit(X_train_scaled, y_train,
                            epochs=5,
                            batch_size=32,
                            validation_data=(X_val, y_val)
                           )

Epoch 1/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 239s 86ms/step - accuracy: 0.8188 - loss: 0.7028 - val_accuracy: 0.8276 - val_loss: 0.6617
Epoch 2/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 261s 86ms/step - accuracy: 0.8286 - loss: 0.6622 - val_accuracy: 0.8276 - val_loss: 0.6583
Epoch 3/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 265s 87ms/step - accuracy: 0.8279 - loss: 0.6623 - val_accuracy: 0.8276 - val_loss: 0.6579
Epoch 4/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 258s 85ms/step - accuracy: 0.8284 - loss: 0.6596 - val_accuracy: 0.8276 - val_loss: 0.6542
Epoch 5/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 233s 85ms/step - accuracy: 0.8214 - loss: 0.6704 - val_accuracy: 0.8324 - val_loss: 1.0483


In [ ]:
# Predict on the test set
y_test_pred_rnn = rnn_model.predict(X_test_new).argmax(axis=1)

343/343 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step


In [ ]:
# Evaluate RNN Model on Test Data
rnn_test_loss, rnn_test_acc = rnn_model.evaluate(X_test_new, y_test_new, verbose=0)
print(f"RNN Test Loss: {rnn_test_loss:.4f}")
print(f"RNN Test Accuracy: {rnn_test_acc:.4f}")

RNN Test Loss: 1.0464
RNN Test Accuracy: 0.8339


In [ ]:
# Classification report for RNN
print("RNN Classification Report:")
print(classification_report(y_test_new, y_test_pred_rnn))

RNN Classification Report:
              precision    recall  f1-score   support

         0.0       0.84      0.98      0.91      9059
         1.0       0.00      0.00      0.00       278
         2.0       0.58      0.30      0.39       724
         3.0       0.00      0.00      0.00        81
         4.0       0.00      0.00      0.00       804

    accuracy                           0.83     10946
   macro avg       0.28      0.26      0.26     10946
weighted avg       0.74      0.83      0.78     10946



# GRU

In [ ]:
def build_gru_model():
    model = Sequential()
    model.add(GRU(64, input_shape=(X_train.shape[1], 1), return_sequences=True))
    model.add(Dropout(0.2))
    model.add(GRU(32))
    model.add(Dropout(0.2))
    model.add(Dense(5, activation='softmax'))

    return model

In [ ]:
gru_model = build_gru_model()

In [ ]:
gru_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

In [ ]:
gru_history = gru_model.fit(X_train_scaled, y_train,
                            epochs=5,
                            batch_size=32,
                            validation_data=(X_val, y_val)
                            )

Epoch 1/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 590s 214ms/step - accuracy: 0.8352 - loss: 0.6585 - val_accuracy: 0.9218 - val_loss: 0.3226
Epoch 2/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 587s 214ms/step - accuracy: 0.9303 - loss: 0.2889 - val_accuracy: 0.9498 - val_loss: 0.2091
Epoch 3/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 622s 214ms/step - accuracy: 0.9461 - loss: 0.2055 - val_accuracy: 0.9392 - val_loss: 0.2036
Epoch 4/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 610s 210ms/step - accuracy: 0.9566 - loss: 0.1596 - val_accuracy: 0.9636 - val_loss: 0.1376
Epoch 5/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 616s 208ms/step - accuracy: 0.9652 - loss: 0.1301 - val_accuracy: 0.9678 - val_loss: 0.1253


In [ ]:
y_test_pred_gru = gru_model.predict(X_test_new).argmax(axis=1)

343/343 ━━━━━━━━━━━━━━━━━━━━ 15s 42ms/step


In [ ]:
# Evaluate GRU Model on Test Data
gru_test_loss, gru_test_acc = gru_model.evaluate(X_test_new, y_test_new, verbose=0)
print(f"GRU Test Loss: {gru_test_loss:.4f}")
print(f"GRU Test Accuracy: {gru_test_acc:.4f}")

GRU Test Loss: 0.1152
GRU Test Accuracy: 0.9699


In [ ]:
# GRU classification report
print("GRU Classification Report:")
print(classification_report(y_test_new, y_test_pred_gru))

GRU Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.98      9059
         1.0       0.95      0.55      0.70       278
         2.0       0.91      0.90      0.90       724
         3.0       0.86      0.46      0.60        81
         4.0       0.97      0.96      0.96       804

    accuracy                           0.97     10946
   macro avg       0.93      0.77      0.83     10946
weighted avg       0.97      0.97      0.97     10946



# LSTM

In [ ]:
def build_lstm_model():
    model = Sequential()
    model.add(LSTM(64, input_shape=(X_train.shape[1], 1), return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(32))
    model.add(Dropout(0.2))
    model.add(Dense(5, activation='softmax'))

    return model

In [ ]:
lstm_model = build_lstm_model()

In [ ]:
lstm_model.compile(optimizer='adam',
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])

In [ ]:
lstm_history = lstm_model.fit(X_train_scaled, y_train,
                            epochs=5,
                            batch_size=32,
                            validation_data=(X_val, y_val)
                            )

Epoch 1/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 430s 156ms/step - accuracy: 0.8264 - loss: 0.7009 - val_accuracy: 0.8276 - val_loss: 0.6598
Epoch 2/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 435s 159ms/step - accuracy: 0.8286 - loss: 0.6545 - val_accuracy: 0.8276 - val_loss: 0.6127
Epoch 3/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 435s 156ms/step - accuracy: 0.8361 - loss: 0.5641 - val_accuracy: 0.8702 - val_loss: 0.4364
Epoch 4/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 443s 157ms/step - accuracy: 0.8950 - loss: 0.3905 - val_accuracy: 0.9017 - val_loss: 0.3649
Epoch 5/5
2737/2737 ━━━━━━━━━━━━━━━━━━━━ 436s 154ms/step - accuracy: 0.9096 - loss: 0.3362 - val_accuracy: 0.9117 - val_loss: 0.3452


In [ ]:
y_test_pred_lstm = lstm_model.predict(X_test_new).argmax(axis=1)

343/343 ━━━━━━━━━━━━━━━━━━━━ 18s 52ms/step


In [ ]:
# Evaluate LSTM Model on Test Data
lstm_test_loss, lstm_test_acc = lstm_model.evaluate(X_test_new, y_test_new, verbose=0)
print(f"LSTM Test Loss: {lstm_test_loss:.4f}")
print(f"LSTM Test Accuracy: {lstm_test_acc:.4f}")

LSTM Test Loss: 0.3416
LSTM Test Accuracy: 0.9113


In [ ]:
# LSTM classification report
print("LSTM Classification Report:")
print(classification_report(y_test_new, y_test_pred_lstm))

LSTM Classification Report:
              precision    recall  f1-score   support

         0.0       0.95      0.96      0.96      9059
         1.0       0.93      0.05      0.10       278
         2.0       0.57      0.76      0.65       724
         3.0       0.00      0.00      0.00        81
         4.0       0.83      0.86      0.85       804

    accuracy                           0.91     10946
   macro avg       0.66      0.53      0.51     10946
weighted avg       0.91      0.91      0.90     10946

